# Denoising Diffusion Probabilistic Model (DDPM) on MNIST
Implement a DDPM similar to the Jackson-Kang tutorial, train on MNIST, sample new images, and evaluate with SSIM and FID.

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
import torchvision.transforms as transforms
from torchvision.utils import make_grid

from tqdm import tqdm

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure

In [ ]:
seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

data_dir = "./data"

img_size = (28, 28, 1)
timestep_embedding_dim = 256
n_layers = 8
hidden_dim = 256
n_timesteps = 1000
beta_minmax = [1e-4, 2e-2]

train_batch_size = 128
inference_batch_size = 64
epochs = 20
lr = 2e-4

eval_num_samples = 10000  # Lower this if you need faster evaluation.

hidden_dims = [hidden_dim for _ in range(n_layers)]

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

num_workers = 0
pin_memory = device.type == "cuda"

train_dataset = MNIST(data_dir, transform=transform, train=True, download=True)
test_dataset = MNIST(data_dir, transform=transform, train=False, download=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True,
    )
test_loader = DataLoader(
    test_dataset,
    batch_size=inference_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    )

## Model and diffusion process
The model mirrors the tutorial: a stacked convolutional denoiser with sinusoidal timestep embeddings and a Gaussian diffusion process.

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


class ConvBlock(nn.Conv2d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        activation_fn=None,
        drop_rate=0.0,
        stride=1,
        padding="same",
        dilation=1,
        groups=1,
        bias=True,
        gn=False,
        gn_groups=8,
    ):
        if padding == "same":
            padding = kernel_size // 2 * dilation

        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            groups=groups,
            bias=bias,
        )

        self.activation_fn = nn.SiLU() if activation_fn else None
        self.group_norm = nn.GroupNorm(gn_groups, out_channels) if gn else None

    def forward(self, x, time_embedding=None, residual=False):
        if residual:
            x = x + time_embedding
            y = x
            x = super().forward(x)
            y = y + x
        else:
            y = super().forward(x)
        y = self.group_norm(y) if self.group_norm is not None else y
        y = self.activation_fn(y) if self.activation_fn is not None else y
        return y


class Denoiser(nn.Module):
    def __init__(
        self,
        image_resolution,
        hidden_dims=None,
        diffusion_time_embedding_dim=256,
        n_times=1000,
    ):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = [256, 256]

        _, _, img_C = image_resolution
        self.time_embedding = SinusoidalPosEmb(diffusion_time_embedding_dim)

        self.in_project = ConvBlock(img_C, hidden_dims[0], kernel_size=7)
        self.time_project = nn.Sequential(
            ConvBlock(diffusion_time_embedding_dim, hidden_dims[0], kernel_size=1, activation_fn=True),
            ConvBlock(hidden_dims[0], hidden_dims[0], kernel_size=1),
        )

        self.convs = nn.ModuleList(
            [ConvBlock(in_channels=hidden_dims[0], out_channels=hidden_dims[0], kernel_size=3)]
        )
        for idx in range(1, len(hidden_dims)):
            self.convs.append(
                ConvBlock(
                    hidden_dims[idx - 1],
                    hidden_dims[idx],
                    kernel_size=3,
                    dilation=3 ** ((idx - 1) // 2),
                    activation_fn=True,
                    gn=True,
                    gn_groups=8,
                )
            )

        self.out_project = ConvBlock(hidden_dims[-1], out_channels=img_C, kernel_size=3)

    def forward(self, perturbed_x, diffusion_timestep):
        y = perturbed_x
        diffusion_embedding = self.time_embedding(diffusion_timestep)
        diffusion_embedding = self.time_project(diffusion_embedding.unsqueeze(-1).unsqueeze(-2))
        y = self.in_project(y)
        for conv in self.convs:
            y = conv(y, diffusion_embedding, residual=True)
        y = self.out_project(y)
        return y

In [ ]:
class Diffusion(nn.Module):
    def __init__(self, model, image_resolution, n_times=1000, beta_minmax=None, device="cuda"):
        super().__init__()
        if beta_minmax is None:
            beta_minmax = [1e-4, 2e-2]

        self.n_times = n_times
        self.img_H, self.img_W, self.img_C = image_resolution
        self.model = model
        self.device = device

        beta_1, beta_T = beta_minmax
        betas = torch.linspace(start=beta_1, end=beta_T, steps=n_times, device=device)
        self.sqrt_betas = torch.sqrt(betas)

        self.alphas = 1 - betas
        self.sqrt_alphas = torch.sqrt(self.alphas)
        alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1 - alpha_bars)
        self.sqrt_alpha_bars = torch.sqrt(alpha_bars)

    def extract(self, a, t, x_shape):
        b, *_ = t.shape
        out = a.gather(-1, t)
        return out.reshape(b, *((1,) * (len(x_shape) - 1)))

    def scale_to_minus_one_to_one(self, x):
        return x * 2 - 1

    def reverse_scale_to_zero_to_one(self, x):
        return (x + 1) * 0.5

    def make_noisy(self, x_zeros, t):
        epsilon = torch.randn_like(x_zeros, device=self.device)
        sqrt_alpha_bar = self.extract(self.sqrt_alpha_bars, t, x_zeros.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, t, x_zeros.shape)
        noisy_sample = x_zeros * sqrt_alpha_bar + epsilon * sqrt_one_minus_alpha_bar
        return noisy_sample.detach(), epsilon

    def forward(self, x_zeros):
        x_zeros = self.scale_to_minus_one_to_one(x_zeros)
        b, _, _, _ = x_zeros.shape
        t = torch.randint(low=0, high=self.n_times, size=(b,), device=self.device).long()
        perturbed_images, epsilon = self.make_noisy(x_zeros, t)
        pred_epsilon = self.model(perturbed_images, t)
        return perturbed_images, epsilon, pred_epsilon

    def denoise_at_t(self, x_t, timestep, t):
        if t > 1:
            z = torch.randn_like(x_t, device=self.device)
        else:
            z = torch.zeros_like(x_t, device=self.device)

        epsilon_pred = self.model(x_t, timestep)
        alpha = self.extract(self.alphas, timestep, x_t.shape)
        sqrt_alpha = self.extract(self.sqrt_alphas, timestep, x_t.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, timestep, x_t.shape)
        sqrt_beta = self.extract(self.sqrt_betas, timestep, x_t.shape)
        x_t_minus_1 = (
            1 / sqrt_alpha
            * (x_t - (1 - alpha) / sqrt_one_minus_alpha_bar * epsilon_pred)
            + sqrt_beta * z
        )
        return x_t_minus_1.clamp(-1.0, 1.0)

    def sample(self, N):
        x_t = torch.randn((N, self.img_C, self.img_H, self.img_W), device=self.device)
        for t in range(self.n_times - 1, -1, -1):
            timestep = torch.full((N,), t, device=self.device, dtype=torch.long)
            x_t = self.denoise_at_t(x_t, timestep, t)
        x_0 = self.reverse_scale_to_zero_to_one(x_t)
        return x_0

    def sample_with_intermediates(self, N, num_steps=8):
        x_t = torch.randn((N, self.img_C, self.img_H, self.img_W), device=self.device)
        intermediates = []
        capture_steps = np.linspace(self.n_times - 1, 0, num_steps, dtype=int).tolist()
        for t in range(self.n_times - 1, -1, -1):
            timestep = torch.full((N,), t, device=self.device, dtype=torch.long)
            x_t = self.denoise_at_t(x_t, timestep, t)
            if t in capture_steps:
                intermediates.append(self.reverse_scale_to_zero_to_one(x_t.detach().clone()))
        return intermediates

In [ ]:
model = Denoiser(
    image_resolution=img_size,
    hidden_dims=hidden_dims,
    diffusion_time_embedding_dim=timestep_embedding_dim,
    n_times=n_timesteps,
).to(device)

diffusion = Diffusion(
    model,
    image_resolution=img_size,
    n_times=n_timesteps,
    beta_minmax=beta_minmax,
    device=device,
).to(device)

optimizer = Adam(diffusion.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
denoising_loss = nn.MSELoss()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Number of model parameters:", count_parameters(diffusion))

## Training
The model predicts the noise added at timestep $t$ and is trained with MSE loss.

In [ ]:
epoch_losses = []
if device.type == "cuda":
    torch.cuda.synchronize()
start_time = time.perf_counter()

for epoch in range(epochs):
    diffusion.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
    for x, _ in progress:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        _, epsilon, pred_epsilon = diffusion(x)
        loss = denoising_loss(pred_epsilon, epsilon)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        progress.set_postfix(loss=loss.item())
    avg_loss = running_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch + 1}: loss={avg_loss:.6f}")

if device.type == "cuda":
    torch.cuda.synchronize()
training_time_s = time.perf_counter() - start_time
best_loss = float(np.min(epoch_losses))
print(f"Training finished in {training_time_s:.2f}s. Best loss: {best_loss:.6f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Noise Prediction Loss (MSE)")
plt.title("DDPM Training Loss")
plt.grid(True)
plt.show()

## Sampling
Generate new images by reversing the diffusion process from pure noise.

In [ ]:
diffusion.eval()
with torch.no_grad():
    generated_images = diffusion.sample(N=inference_batch_size)

grid = make_grid(generated_images, nrow=8, padding=2, normalize=True)
plt.figure(figsize=(6, 6))
plt.axis("off")
plt.title("DDPM Samples")
plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
plt.show()

In [ ]:
with torch.no_grad():
    intermediates = diffusion.sample_with_intermediates(N=1, num_steps=8)

fig, axes = plt.subplots(1, len(intermediates), figsize=(12, 2))
for ax, img in zip(axes, intermediates):
    ax.imshow(img[0].squeeze(0).cpu().numpy(), cmap="gray")
    ax.axis("off")
plt.suptitle("Reverse diffusion steps")
plt.show()

## Evaluation metrics
Compute SSIM and FID between generated samples and MNIST test images.

In [ ]:
eval_num_samples = min(eval_num_samples, len(test_dataset))

def collect_real_images(loader, num_samples):
    images = []
    total = 0
    for x, _ in loader:
        images.append(x)
        total += x.size(0)
        if total >= num_samples:
            break
    return torch.cat(images, dim=0)[:num_samples]

real_images = collect_real_images(test_loader, eval_num_samples)
perm = torch.randperm(real_images.size(0))
real_images = real_images[perm]

diffusion.eval()
fake_images = []
with torch.no_grad():
    for i in range(0, eval_num_samples, inference_batch_size):
        cur_batch = min(inference_batch_size, eval_num_samples - i)
        fake_batch = diffusion.sample(N=cur_batch)
        fake_images.append(fake_batch.cpu())
fake_images = torch.cat(fake_images, dim=0)

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
for i in range(0, eval_num_samples, inference_batch_size):
    real_batch = real_images[i : i + inference_batch_size].to(device)
    fake_batch = fake_images[i : i + inference_batch_size].to(device)
    ssim_metric.update(fake_batch, real_batch)
ssim_score = float(ssim_metric.compute().cpu().item())
ssim_metric.reset()

def preprocess_for_fid(x):
    if x.size(1) == 1:
        x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
    x = (x * 255).clamp(0, 255).to(torch.uint8)
    return x

fid = FrechetInceptionDistance(feature=2048).to(device)

def update_fid(images, real):
    for i in range(0, images.size(0), inference_batch_size):
        batch = images[i : i + inference_batch_size].to(device)
        batch = preprocess_for_fid(batch)
        fid.update(batch, real=real)

update_fid(real_images, real=True)
update_fid(fake_images, real=False)
fid_score = float(fid.compute().cpu().item())

print(f"SSIM: {ssim_score:.4f} | FID: {fid_score:.4f}")

In [ ]:
results = pd.DataFrame(
    {
        "Best Loss (MSE)": [best_loss],
        "Training Time (s)": [training_time_s],
        "SSIM": [ssim_score],
        "FID": [fid_score],
    },
    index=["Table 1"],
)
results